# Imports

In [1]:
import os
import pandas as pd
import numpy as np
import glob
import warnings
import re
import datetime


In [2]:
from pathlib import Path
from datetime import datetime

# Processing

## Configuration

In [ ]:
ROOT_FOLDER   = r"" #Upload the input file path here
output_folder = r"" #Upload the output main file path here
versioned_folder = r"" #Upload the version files path here

## Mapping

In [4]:
ENROLLMENT_MAP = {
    "continuing"              : "Continuing",
    "continuingcap"           : "ContinuingCAP",
    "continuing cap"          : "ContinuingCAP",
    "continuingnon-matric"    : "ContinuingNon-Matric",
    "continuing non-matric"   : "ContinuingNon-Matric",
    "continuing non matric"   : "ContinuingNon-Matric",
    "newcap"                  : "NewCAP",
    "new cap"                 : "NewCAP",
    "new first-time"          : "NewFirst-Time",
    "new first time"          : "NewFirst-Time",
    "newfirst-time"           : "NewFirst-Time",
    "newnon-matric"           : "NewNon-Matric",
    "new non-matric"          : "NewNon-Matric",
    "new non matric"          : "NewNon-Matric",
    "newtransfer"             : "NewTransfer",
    "new transfer"            : "NewTransfer",
    "continuingptech"         : "ContinuingPTech",
    "continuing ptech"        : "ContinuingPTech",
    "crossregistered"         : "CrossRegistered",
    "cross registered"        : "CrossRegistered",
    "cross-registered"        : "CrossRegistered",
}

## Pipeline

In [5]:
def extract_metadata(raw_df):
    term, run_date = None, None
    for col in range(raw_df.shape[1]):
        cell = str(raw_df.iloc[0, col])
        if not term:
            m = re.search(r'Term[:\s]+(\S+)', cell, re.IGNORECASE)
            if m:
                term = m.group(1)
        if not run_date:
            m = re.search(r'Run\s*Date[:\s]+([\d/]+\s+[\d:]+\s*(?:AM|PM)?)', cell, re.IGNORECASE)
            if m:
                run_date = m.group(1).strip()
    return term, run_date


def normalise(label):
    return str(label).strip().lower().replace("  ", " ")


def is_valid_label(label):
    return str(label).strip().lower() not in ('', 'nan', 'none', 'null')


def parse_fte_sheet(filepath, sheet_name):
    """
    Reads one FTE sheet and returns a LONG-FORMAT DataFrame.
    Dynamically adapts to missing columns, injects zeros, 
    and securely extracts the Run Date from the FILENAME to prevent human error!
    """
    filename = Path(filepath).name
    
    # --- 1. EXTRACT DATE FROM FILENAME (Ultimate Source of Truth) ---
    true_date = None
    date_match = re.search(r'^(\d{4})\s*\.\s*(\d{1,2})\s*\.\s*(\d{1,2})', filename)
    if date_match:
        y, m, d = date_match.groups()
        try:
            # THE FIX: Create the datetime object first, then pull the date!
            true_date = datetime(int(y), int(m), int(d)).date()
        except ValueError:
            pass

    # --- 2. Extract Term from inside the sheet ---
    raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    term, internal_date_str = extract_metadata(raw)

    # --- 3. Lock in the Run Date ---
    if true_date:
        run_date = true_date
    else:
        run_date = pd.to_datetime(internal_date_str, errors='coerce')
        if pd.notna(run_date):
            run_date = run_date.date()

    # --- DYNAMIC HEADER DETECTION ---
    header_row_idx = 2  # Fallback
    for i in range(min(10, len(raw))):
        cell_val = str(raw.iloc[i, 0]).strip().lower()
        if 'enrollment' in cell_val and 'status' in cell_val:
            header_row_idx = i
            break

    hdr_series = raw.iloc[header_row_idx].ffill()
    
    col_names = []
    counts = {'FullTime': 0, 'PartTime': 0, 'GrandTotal': 0, 'Unknown': 0}
    suffixes = ['_Headcount', '_Credits', '_FTE']

    for val in hdr_series:
        v_str = str(val).lower().replace('\n', ' ')
        if 'enrollment' in v_str:
            col_names.append('Enrollment_Status')
        elif 'full' in v_str:
            base = 'FullTime'
            idx = counts[base]
            col_names.append(base + suffixes[idx] if idx < 3 else f"{base}_{idx}")
            counts[base] += 1
        elif 'part' in v_str:
            base = 'PartTime'
            idx = counts[base]
            col_names.append(base + suffixes[idx] if idx < 3 else f"{base}_{idx}")
            counts[base] += 1
        elif 'total' in v_str:
            base = 'GrandTotal'
            idx = counts[base]
            col_names.append(base + suffixes[idx] if idx < 3 else f"{base}_{idx}")
            counts[base] += 1
        else:
            col_names.append(f"Drop_{counts['Unknown']}")
            counts['Unknown'] += 1

    # --- DATA EXTRACTION ---
    data = raw.iloc[header_row_idx + 2:].copy()
    data.columns = col_names[:data.shape[1]]
    data = data.loc[:, ~data.columns.str.startswith('Drop_')]

    data.dropna(how='all', inplace=True)
    data.dropna(subset=['Enrollment_Status'], inplace=True)
    data.reset_index(drop=True, inplace=True)

    for col in data.columns:
        if col != 'Enrollment_Status':
            data[col] = pd.to_numeric(data[col], errors='coerce')

    data = data[data['Enrollment_Status'].apply(is_valid_label)].copy()
    data.reset_index(drop=True, inplace=True)

    is_total = data['Enrollment_Status'].astype(str).str.strip().str.lower() == 'total'
    detail   = data[~is_total].copy()

    # --- BUILD FINAL ROWS ---
    long_rows = []
    for _, row in detail.iterrows():
        label = row['Enrollment_Status']
        if not is_valid_label(label):
            continue

        enrollment_status = str(label).strip()

        ft_hc = row.get('FullTime_Headcount')
        ft_cr = row.get('FullTime_Credits')
        ft_fte = row.get('FullTime_FTE')

        long_rows.append({
            'Source_File'       : filename,
            'Term'              : term,
            'Run_Date'          : run_date,
            'Enrollment_Status' : enrollment_status,
            'Type'              : 'Full Time',
            'Headcount'         : 0 if pd.isna(ft_hc) else ft_hc,
            'Credits'           : 0 if pd.isna(ft_cr) else ft_cr,
            'FTE'               : 0 if pd.isna(ft_fte) else ft_fte,
        })

        pt_hc = row.get('PartTime_Headcount')
        pt_cr = row.get('PartTime_Credits')
        pt_fte = row.get('PartTime_FTE')

        long_rows.append({
            'Source_File'       : filename,
            'Term'              : term,
            'Run_Date'          : run_date,
            'Enrollment_Status' : enrollment_status,
            'Type'              : 'Part Time',
            'Headcount'         : 0 if pd.isna(pt_hc) else pt_hc,
            'Credits'           : 0 if pd.isna(pt_cr) else pt_cr,
            'FTE'               : 0 if pd.isna(pt_fte) else pt_fte,
        })

    return pd.DataFrame(long_rows)


def find_fte_sheet(xl):
    for name in xl.sheet_names:
        if name.upper().startswith('FTE'):
            return name
    return None


def read_all_fte(root_folder):
    root       = Path(root_folder)
    all_frames = []
    EXCEL_EXTS = {'.xlsx', '.xls', '.xlsm', '.xlsb'}

    for year_dir in sorted(root.iterdir()):
        if not year_dir.is_dir():
            continue
        year = year_dir.name

        for term_dir in sorted(year_dir.iterdir()):
            if not term_dir.is_dir():
                continue

            for wb_path in sorted(term_dir.glob('*')):
                if wb_path.suffix.lower() not in EXCEL_EXTS:
                    continue
                if wb_path.stem.startswith('~'):
                    continue

                print(f"  Reading : {wb_path.relative_to(root)}")
                try:
                    xl         = pd.ExcelFile(wb_path)
                    sheet_name = find_fte_sheet(xl)
                    if sheet_name is None:
                        print("    WARNING: No FTE sheet found - skipped")
                        continue

                    df = parse_fte_sheet(wb_path, sheet_name)
                    if not df.empty:
                        all_frames.append(df)

                except Exception as e:
                    print(f"    ERROR reading {wb_path.name}: {e}")

    if not all_frames:
        return pd.DataFrame()

    df_all = pd.concat(all_frames, ignore_index=True)

    # Sort by Term → Run_Date ascending (earliest run date first within each term)
    df_all['_rd'] = pd.to_datetime(df_all['Run_Date'], errors='coerce')
    df_all.sort_values(['Term', '_rd'], ascending=[True, True], inplace=True)
    df_all.drop(columns=['_rd'], inplace=True)
    df_all.reset_index(drop=True, inplace=True)

    return df_all


## Reading Files 

In [ ]:
df_summary = read_all_fte(ROOT_FOLDER)

## Exporting

In [ ]:
if __name__ == '__main__':
    
    print(f"Scanning: {ROOT_FOLDER}\n")
    _now = datetime.now()
    _ampm = "A.M" if _now.strftime("%p") == "AM" else "P.M"
    run_timestamp = (f"{_now.year}.{_now.month}.{_now.day}_"
                    f"{_now.strftime('%I').lstrip('0')}.{_now.strftime('%M')}.{_ampm}")

    # --- Path 1: Overwrite existing file (live dashboard) ---
    live_output_path = Path(output_folder) / "FTE_Combined_Output.xlsx"

    # --- Path 2: Versioned file with run date/time stamp ---
    versioned_filename = f"Dashboard_data_{run_timestamp}.xlsx"
    versioned_output_path = Path(versioned_folder) / versioned_filename

    def write_excel(path):
        with pd.ExcelWriter(path, engine='openpyxl', date_format='M/D/YYYY') as writer:
            df_summary.to_excel(writer, sheet_name='Summary', index=False)

            if not df_summary.empty and 'Run_Date' in df_summary.columns:
                df_summary['_year'] = pd.to_datetime(df_summary['Run_Date'], errors='coerce').dt.year
                for yr, grp in df_summary.groupby('_year'):
                    grp.drop(columns=['_year']).to_excel(writer, sheet_name=str(yr)[:31], index=False)
                df_summary.drop(columns=['_year'], inplace=True)

    write_excel(live_output_path)
    write_excel(versioned_output_path)

    print(f"\nLive file updated : {live_output_path}")
    print(f"Version saved     : {versioned_output_path}")
    print(f"Total rows        : {len(df_summary)}")
    print(f"\nPreview:\n{df_summary.head(10).to_string(index=False)}")